In [1]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)


device: mps


In [2]:
model_name = "typeform/distilbert-base-uncased-mnli"

classifier = pipeline(
    "zero-shot-classification",
    model=model_name,
    device=device,
)

print(model_name)


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

typeform/distilbert-base-uncased-mnli


In [3]:
ds = load_dataset("glue", "mrpc", split="validation")
print(ds)
print(ds[0])

sent1 = ds["sentence1"]
sent2 = ds["sentence2"]
y_true = np.array(ds["label"])
texts = [f"Sentence 1: {s1}\nSentence 2: {s2}" for s1, s2 in zip(sent1, sent2)]

print("num_examples:", len(y_true))
print("positive_rate:", y_true.mean())
print(texts[0])


Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 408
})
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}
num_examples: 408
positive_rate: 0.6838235294117647
Sentence 1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
Sentence 2: " The foodservice pie business does not fit our long-term growth strategy .


In [4]:
candidate_labels = ["same meaning", "different meaning"]
hypothesis_template = "The two sentences have {}."

batch_size = 32
raw_outputs = []
y_pred = []
score_dicts = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch_texts = texts[i:i + batch_size]
    batch_outputs = classifier(
        batch_texts,
        candidate_labels=candidate_labels,
        hypothesis_template=hypothesis_template,
        multi_label=False,
        batch_size=batch_size,
        truncation=True,
        max_length=256,
    )
    if isinstance(batch_outputs, dict):
        batch_outputs = [batch_outputs]

    for out in batch_outputs:
        raw_outputs.append(out)
        scores = {label: float(score) for label, score in zip(out["labels"], out["scores"])}
        same_score = scores.get("same meaning", 0.0)
        diff_score = scores.get("different meaning", 0.0)
        score_dicts.append({
            "same meaning": same_score,
            "different meaning": diff_score,
        })
        y_pred.append(1 if out["labels"][0] == "same meaning" else 0)

y_pred = np.array(y_pred)
print("done")
print(raw_outputs[0])


  0%|          | 0/13 [00:00<?, ?it/s]

done
{'sequence': 'Sentence 1: He said the foodservice pie business doesn \'t fit the company \'s long-term growth strategy .\nSentence 2: " The foodservice pie business does not fit our long-term growth strategy .', 'labels': ['different meaning', 'same meaning'], 'scores': [0.9994259476661682, 0.000574022124055773]}


In [5]:
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print({"accuracy": acc, "f1": f1})
print(classification_report(y_true, y_pred, target_names=["different meaning", "same meaning"]))


{'accuracy': 0.4117647058823529, 'f1': 0.28994082840236685}
                   precision    recall  f1-score   support

different meaning       0.34      0.92      0.50       129
     same meaning       0.83      0.18      0.29       279

         accuracy                           0.41       408
        macro avg       0.59      0.55      0.39       408
     weighted avg       0.68      0.41      0.36       408



In [6]:
for i in range(5):
    print("=" * 80)
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]))
    print("same meaning score:", round(score_dicts[i]["same meaning"], 6))
    print("different meaning score:", round(score_dicts[i]["different meaning"], 6))

mistakes = np.where(y_true != y_pred)[0][:10]
print("num_errors:", int((y_true != y_pred).sum()))

for i in mistakes:
    print("=" * 80)
    print("idx:", int(i))
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]))
    print("same meaning score:", round(score_dicts[i]["same meaning"], 6))
    print("different meaning score:", round(score_dicts[i]["different meaning"], 6))


sentence1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
sentence2: " The foodservice pie business does not fit our long-term growth strategy .
true: 1 pred: 0
same meaning score: 0.000574
different meaning score: 0.999426
sentence1: Magnarelli said Racicot hated the Iraqi regime and looked forward to using his long years of training in the war .
sentence2: His wife said he was " 100 percent behind George Bush " and looked forward to using his years of training in the war .
true: 0 pred: 0
same meaning score: 0.341497
different meaning score: 0.658503
sentence1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .
sentence2: The dollar was at 116.78 yen JPY = , virtually flat on the session , and at 1.2871 against the Swiss franc CHF = , down 0.1 percent .
true: 0 pred: 0
same meaning score: 0.47575
different meaning score: 0.52425
sentence1: The AFL-CIO is waiting until O

In [7]:
summary = {
    "dataset": "glue/mrpc",
    "split": "validation",
    "model": model_name,
    "device": str(device),
    "num_examples": len(ds),
    "accuracy": float(acc),
    "f1": float(f1),
}
summary


{'dataset': 'glue/mrpc',
 'split': 'validation',
 'model': 'typeform/distilbert-base-uncased-mnli',
 'device': 'mps',
 'num_examples': 408,
 'accuracy': 0.4117647058823529,
 'f1': 0.28994082840236685}